In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from goad_toolkit.visualizer import CorrelationHeatmap, PlotSettings, RegPlot, ScatterPlot
from wa_analyzer.data import load_showcase

import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)

df = load_showcase("penguins_raw")
select = [
    "Species",
    "Island",
    "Culmen Length (mm)",
    "Culmen Depth (mm)",
    "Flipper Length (mm)",
    "Delta 15 N (o/oo)",
    "Delta 13 C (o/oo)",
    "Sex",
    "Body Mass (g)",
]
subset = df[select].dropna()
subset["species"] = subset["Species"].str.split(" ").str[0]
subset.species.value_counts()

# Correlatie

Correlatie is een maatstaf voor de samenhang tussen twee variabelen. De meest gebruikte maatstaf in de statistiek is de Pearson-correlatiecoëfficiënt, die de lineaire relatie tussen twee variabelen meet.

Het idee is als volgt: we hebben gekeken naar variantie, die het verschil van elk individueel datapunt ten opzichte van het gemiddelde berekent via $(x_i - \bar{x})^2$ en daar het gemiddelde van neemt. Wat als we deze afwijkingen tussen twee variabelen met elkaar vergelijken? Dat is wat de covariantie doet:

$$cov(x,y) = \frac{1}{n} \sum_i (x_i - \bar{x})(y_i - \bar{y})$$

Dit is lastig te interpreteren, omdat het afhangt van de meeteenheden van de variabelen. We kunnen dit normaliseren door te delen door het product van de standaarddeviaties van beide variabelen:

$$ r = \frac{cov(x,y)}{\sigma_x \sigma_y}$$

waarbij $\sigma$ de standaarddeviatie is.

Of, indien je wilt corrigeren voor steekproefbias:

$$r = \frac{cov(x,y)}{(n-1)\sigma_x \sigma_y}$$

Dit is de Pearson-correlatiecoëfficiënt: een getal tussen -1 en 1, waarbij 1 een perfecte positieve correlatie weergeeft, 0 geen correlatie, en -1 een perfecte negatieve correlatie.

In [ ]:
floats = subset.select_dtypes(include="float64")
correlation_matrix = floats.corr()
correlation_matrix.round(2)

In [ ]:
settings = PlotSettings(
    figsize=(8, 6),
    title="Six measurements, fifteen correlations",
    xlabel="",
    ylabel="",
)
fig, ax = CorrelationHeatmap(settings).plot(data=floats)

## De variabele die niet in het plaatje zit

Kijk eens naar de regel die de matrix opbouwde:

```python
floats = subset.select_dtypes(include="float64")
```

`Species`, `Island` en `Sex` zijn tekst, dus die werden weggegooid vóórdat er ook maar één correlatie werd berekend. De heatmap kan je niets laten zien wat hij nooit heeft ontvangen — en elk van die vijftien getallen werd gemeten over drie soorten die op één hoop waren gegooid.

Vraag het dus aan elk paar twee keer: één keer samengevoegd (*pooled*), en één keer **binnen** (*within*) elke soort. Het gemiddelde van elke soort aftrekken vóórdat je correleert is alles wat nodig is — het verwijdert de verschillen *tussen* de soorten en behoudt de verschillen *binnen* de soorten.

In [ ]:
def within_species(column: str) -> pd.Series:
    """The column with each species' own mean removed."""
    return subset[column] - subset.groupby("species")[column].transform("mean")


pairs = []
columns = list(floats.columns)
for i, a in enumerate(columns):
    for b in columns[i + 1:]:
        pairs.append({
            "pair": f"{a.split(' (')[0]} / {b.split(' (')[0]}",
            "pooled": subset[a].corr(subset[b]),
            "within species": within_species(a).corr(within_species(b)),
        })

pairs = pd.DataFrame(pairs)
pairs["changes sign"] = np.sign(pairs.pooled) != np.sign(pairs["within species"])
print(f"{pairs['changes sign'].sum()} of the {len(pairs)} correlations change sign")
pairs.sort_values("pooled", key=abs, ascending=False).round(2)

**Zeven van de vijftien wisselen van teken.** Eentje daarvan is een wiebeling tussen twee getallen die in beide gevallen dicht bij nul liggen; de overige zes niet — die veranderen echt van richting, waardoor de conclusie die je op basis van de heatmap zou hebben getrokken exact het tegenovergestelde is van wat er binnen elke afzonderlijke groep gebeurt.

Twee paren zijn de moeite waard om zorgvuldig te bekijken. Begin met de snavels.

In [ ]:
x, y = "Culmen Length (mm)", "Culmen Depth (mm)"

bills = PlotSettings(
    figsize=(12, 4.5),  # ty: ignore[invalid-argument-type]
    title="The same 324 penguins, twice",
    subplot_titles=[f"all of them: r = {subset[x].corr(subset[y]):+.2f}",
                    "one species at a time"],
    xlabel=x,
    ylabel=y,
)

host = ScatterPlot(bills)
fig, axes = host.create_figure(n_plots=2)

host.plot_on_axes(RegPlot(bills), axes[0], data=subset, x=x, y=y, ci=None,
                  scatter_kws={"color": "#999999"})
colours = dict(zip(sorted(subset.species.unique()), ["#4c72b0", "#dd8452", "#55a868"]))
for species, group in subset.groupby("species"):
    host.plot_on_axes(RegPlot(bills), axes[1], data=group, x=x, y=y, ci=None,
                      color=colours[species], label=species,
                      scatter_kws={"alpha": 0.6, "color": colours[species]})
axes[1].legend(title="")

Aan de linkerkant: één dalende lijn door alles heen: langere snavels zijn minder diep, `r = -0.22`.
Aan de rechterkant: drie stijgende lijnen. Binnen elke soort is een langere snavel juist een *diepere* snavel, en dat verschil is allerminst marginaal — `+0.40` voor Adelie, `+0.65` voor Chinstrap, `+0.66` voor Gentoo.

Er is niets aan de vogels veranderd. De samengevoegde lijn meet simpelweg het feit dat Gentoos lange, platte snavels hebben en Adelies korte, diepe snavels, wat de drie puntenwolken langs een dalende diagonaal rangschikt. De helling die je ziet is **tussen** soorten; de helling binnen elke soort wijst precies de andere kant op.

Dit is **Simpson's paradox**, die we in les 2 zagen als een tabel met toelatingen tot Berkeley. Hier is het een spreidingsdiagram, wat het mechanisme direct zichtbaar maakt: je kunt met eigen ogen de drie wolken zien waar de lijn doorheen wordt getrokken.

In [ ]:
for a, b in [("Delta 15 N (o/oo)", "Body Mass (g)"),
             ("Culmen Length (mm)", "Body Mass (g)")]:
    print(f"{a.split(' (')[0]} vs {b.split(' (')[0]}")
    print(f"  pooled          r = {subset[a].corr(subset[b]):+.2f}")
    for species, group in subset.groupby("species"):
        print(f"  {species:10s}      r = {group[a].corr(group[b]):+.2f}   (n={len(group)})")
    print(f"  within species  r = {within_species(a).corr(within_species(b)):+.2f}\n")

Het eerste paar is de krachtigste les, omdat de correlatie niet omkeert — zij **verdwijnt**.

Stikstof-15 (δ15N) correleert samengevoegd met lichaamsgewicht op `-0.54`, wat een getal is dat je geneigd zou zijn te rapporteren. Binnen de soorten is het `+0.03`: helemaal niets, in alle drie de groepen. De correlatie werd niet verzwakt door te controleren voor soort, zij *bestond uitsluitend uit* soort. δ15N stijgt met het trofische niveau, de drie soorten eten anders en verschillen toevallig in grootte, en dat is de gehele relatie. Zodra je weet welke soort een pinguïn is, vertelt zijn stikstofwaarde je helemaal niets meer over hoe zwaar hij is.

Het tweede paar laat zien waarom "wantrouw correlaties" niet de juiste les is. Snavellengte afgezet tegen lichaamsgewicht is `+0.59` samengevoegd én `+0.59` binnen de soorten — onaangetast, positief in elke soort. Een grotere vogel heeft een grotere snavel, zowel binnen groepen als over de groepen heen, en geen enkele vorm van conditioneren laat dat verdwijnen.

**Dezelfde tabel, dezelfde methode, drie totaal verschillende uitkomsten.** Dat is het betrouwbaarheidsraster uit [05.1](05.1-relationships%20jeffrey.ipynb) aan het werk: de juiste reactie hangt af van wat je vindt, en je kunt dat niet vooraf kiezen.

### Wat eraan te doen

Drie manieren om met een confounder om te gaan, in oplopende volgorde van wat ze van je vragen:

1. **Rapporteer per groep.** Drie correlaties in plaats van één. Eerlijk, en vaak het volledige antwoord — "binnen elke soort zijn langere snavels dieper" is een echte bevinding, en niet degene die de heatmap voorschotelde.
2. **Haal de groep er eerst uit**, wat de functie `within_species` hierboven deed: trek het groepsgemiddelde af en correleer wat overblijft. Exact dezelfde 'trek-het-saaie-deel-af'-beweging als in les 3, maar dan toegepast op een categorie in plaats van een seizoen.
3. **Neem het op in een model** naast de variabelen die je interesseren, en lees de coëfficiënten af — wat het onderwerp is van het volgende onderdeel, en van de rest van je ML-cursus.

En het deel dat door geen enkele methode wordt opgelost: **de confounder moet wel in je data zitten**. `Species` zat erin, en het herstellen ervan kostte één `groupby`. Als het Palmer-team niet had genoteerd welke vogel welke was, zou elk getal in die heatmap nog net zo fout zijn geweest, en niets in de dataset had je kunnen waarschuwen.

> Dat is het "4e verdieping"-probleem uit [05.1](05.1-relationships%20jeffrey.ipynb) in zijn algemene vorm: *geen mechanisme* betekent niet dat er geen bevinding is, het betekent dat de echte bevinding een variabele is die je nog niet hebt opgeschreven. Hier was het `Species`. In je eigen project zal het iets zijn wat je niet hebt gemeten, en de enige verdediging is hardop vragen vóórdat je de zin opschrijft: *wat is er nog meer waar over de groepen die ik vergelijk?*

## Correlatie vinden via regularisatie
We beginnen met een ongelukkige dataset: 500 datapunten en 100 features. Maar er zit enorm veel ruis in, en heel veel features correleren niet eens met de doelvariabele (target)!

In de praktijk gebeurt dit vaker dan je lief is: je krijgt een overvloed aan features, en je hebt geen idee wat echte correlatie is en wat pure ruis is.

In [ ]:

datafile = Path("../../data/sim/correlation.csv")
df = pd.read_csv(datafile)
df.shape

Je zou kunnen denken: laten we beginnen met het plotten van de correlaties. Omdat we 100 features hebben, is dat te veel voor een heatmap, dus beperken we ons tot de correlatie tussen de features en de target.

In [ ]:
correlation_matrix = df.corr()
sns.scatterplot(x=correlation_matrix.index, y=correlation_matrix.target)
plt.xticks([])

De target heeft vanzelfsprekend een correlatie van 1 met zichzelf. Maar de andere features zijn veel ruiziger. Zes features hebben een correlatie boven 0,2, maar dat is nog steeds niet erg hoog.
Wellicht zijn de twee features met een correlatie boven 0,5 de moeite waard om te bekijken, maar de rest zou met deze methode zomaar ruis kunnen zijn. Omdat dit synthetische data is die ik zelf heb gegenereerd, weet ik zeker dat 10 van de 100 features relevant zijn. Gewone correlatie schiet hier dus tekort om de relevante features op te sporen.

Er zijn echter slimmere manieren om correlatie te achterhalen. Laten we eerst de data splitsen in features `X` en target `y`.

In [ ]:
X = df.drop(columns=["target"]).values
y = df["target"].values

X.shape, y.shape

## Lineaire regressie (mis)bruiken voor feature selectie
Wat we gaan doen, is een lineair regressiemodel construeren voor dit probleem.

Merk op dat in deze strategie ons doel NIET is om per se een lineair regressiemodel te bouwen. Dat zouden we kunnen willen, maar dat is hier niet het punt. We proberen meer grip te krijgen op correlatie en op de vraag welke features relevante informatie bevatten. We hebben misschien al besloten dat we een random forest, een neuraal netwerk of iets anders gaan gebruiken, of misschien weten we het nog niet. Door een lineair regressiemodel te gebruiken, gaan we **regularisatie** inzetten en de gewichten (*weights*) van het model uitlezen.

Het zijn die gewichten waar we werkelijk in geïnteresseerd zijn, omdat ze ons vertellen hoe informatief de features zijn. Als een gewicht nul is, betekent dit dat de feature niet door het model wordt gebruikt. Is een gewicht hoog, dan weegt de feature zwaar mee. Is een gewicht negatief, dan wordt de feature omgekeerd gebruikt.

We veronderstellen dus een relatie zoals deze:

$$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + ... + \beta_{100} x_{100}$$

waarbij $\beta$ de gewichten zijn die we willen leren.

We definiëren een verliesfunctie (*loss function*). Deze functie geeft aan hoe goed of slecht ons model presteert. Hoe beter het model, hoe lager het verlies. De Mean Squared Error (MSE) is een veelgebruikte verliesfunctie die het gemiddelde van de gekwadrateerde verschillen tussen voorspellingen en werkelijke waarden berekent:

$$L = \frac{1}{n} \sum_i (y_i - \hat{y}_i)^2$$

# Regularisatie toepassen
Maar 100 features is veel, zeker vergeleken met 500 datapunten. Daarom gaan we de gewichten in toom houden door een boete (*penalty*) toe te voegen aan de verliesfunctie. Dit heet **regularisatie**.

## L2-regularisatie (Ridge)
Ridge-regressie (vaak aangeduid als "L2") ziet er zo uit:

$$L = \frac{1}{n} \sum_i (y_i - \hat{y}_i)^2 + \alpha \frac{1}{2} \sum_j \beta_j^2$$

Het eerste deel is de normale verliesfunctie; het tweede deel $\alpha \frac{1}{2} \sum_j \beta_j^2$ is de regularisatieboete.
De factor $\frac{1}{2}$ helpt bij het nemen van de afgeleide: zoals je je wellicht herinnert van wiskunde op school, is de afgeleide van $a^2$ gelijk aan $2a$, maar de afgeleide van $\frac{1}{2}a^2$ is $\frac{1}{2} \cdot 2a = a$, wat wiskundig erg prettig uitkomt.

Wat we in feite doen, is het kwadraat van elk gewicht als boete toevoegen. Begrijp je wat dit doet? De gewichten vormen een extra term die we proberen te minimaliseren. Als twee modellen nagenoeg even goed voorspellen, maar één daarvan lagere gewichten heeft, krijgt dat model de voorkeur. $\alpha$ is een hyperparameter die de sterkte van de regularisatie bepaalt: hoe hoger $\alpha$, hoe sterker de gewichten worden ingeperkt.

## L1-regularisatie (Lasso)
Een andere vorm van regularisatie is Lasso (vaak "L1" genoemd):

$$L = \frac{1}{n} \sum_i (y_i - \hat{y}_i)^2 + \alpha \sum_j |\beta_j|$$

Je herkent dezelfde Mean Squared Error, maar nu tellen we de som van de absolute waarden $|\beta_j|$ op als boete ($|x|$ is altijd positief). $\alpha$ bepaalt of deze boete zwaar meeweegt (grote $\alpha$) of nauwelijks invloed heeft (kleine $\alpha$).

## ElasticNet
Het cruciale verschil tussen L1 en L2 is dat L1 de neiging heeft gewichten exact naar nul te drukken (spaarzaamheid / *sparsity*). Daarom wordt L1 veel gebruikt voor feature selectie. Een andere optie is beide strategieën te combineren via ElasticNet:

$$L = \frac{1}{n} \sum_i (y_i - \hat{y}_i)^2 + r\alpha \sum_j |\beta_j| + \alpha \frac{1-r}{2} \sum_j \beta_j^2$$

Ontleed de formule:

- Deel 1 is de gewone verliesfunctie $\frac{1}{n} \sum_i (y_i - \hat{y}_i)^2$.
- Deel 2 is de L1-regularisatie $\alpha \sum_j |\beta_j|$. Als $r = 0$, verdwijnt deze term.
- Deel 3 is de L2-regularisatie $\alpha \frac{1-r}{2} \sum_j \beta_j^2$. Als $r = 1$, verdwijnt deze term.

We kunnen voor $r$ een waarde tussen 0 en 1 kiezen om de twee strategieën te balanceren. Laten we eerst beginnen met standaardwaarden.

In [ ]:
from sklearn.linear_model import SGDRegressor

regressor = SGDRegressor(penalty="elasticnet", random_state=42)
regressor.fit(X, y)
coef = regressor.coef_
plt.plot(coef.T, "o")  # ty: ignore[unresolved-attribute] -- coef_ is set once fit() has run

Dit is een indrukwekkende verbetering ten opzichte van eenvoudige correlatie!

We hebben de optie om verschillende waarden voor $r$ en $\alpha$ te proberen. Hoewel het bouwen en hypertunen van modellen een onderwerp is voor het volgende semester, laat ik het hier alvast zien zodat je er een eerste indruk van krijgt.

In [ ]:
from sklearn.linear_model import SGDRegressor
from sklearn.model_selection import GridSearchCV


l1_ratio = [0.1, 0.15, 0.3, 0.6, 0.75, 0.9, 0.95, 0.99, 1]
alphalist = [0.001, 0.01, 0.1, 1, 10, 100]

regressor = SGDRegressor(penalty="elasticnet", random_state=42)
param_grid = {"alpha":alphalist, "l1_ratio":l1_ratio}

grid = GridSearchCV(
    regressor,
    param_grid=param_grid,
    cv=5,
)


grid.fit(X, y)
best_model = grid.best_estimator_
coef = best_model.coef_
plt.plot(coef.T, "o")

In [ ]:
grid.best_params_

Kijken we naar het resultaat, dan zie je dat de grid-search koos voor `l1_ratio = 1`, wat betekent dat de zoektocht L1-regularisatie als beste strategie vond voor dit probleem.
